In [9]:
import cv2
from tracker import *
from itertools import zip_longest

In [ ]:
tracker=Euclideandisttraker()
# cap=cv2.VideoCapture(r'C:\Users\beher\OneDrive\Desktop\DeepLearning\Computer vision\tracker img\cars.mp4')
cap=cv2.VideoCapture(r'C:\Users\beher\OneDrive\Desktop\DeepLearning\Computer vision\tracker img\highway.mp4')



if not cap.isOpened():
    print("❌ Video could not be opened!")
else:
    print("✅ Video opened successfully")



object_detector=cv2.createBackgroundSubtractorMOG2(history=100,varThreshold=40)



while True:
    ret,frame=cap.read()
    if not ret:
        break

    height,width,_=frame.shape
    # roi=frame[340:720,500:800]
    roi = frame[250:720, 200:1100]

    mask=object_detector.apply(roi)
    _,mask=cv2.threshold(mask,254,255,cv2.THRESH_BINARY)
    contours,_=cv2.findContours(mask,cv2.RETR_TREE,cv2.CHAIN_APPROX_SIMPLE)
    detections=[]

    for cnt in contours:
        area=cv2.contourArea(cnt)
        if area >500:
            x,y,w,h=cv2.boundingRect(cnt)
            detections.append([x,y,w,h])
    
    boxes_ids=tracker.update(detections)

    for box_id in boxes_ids:
        x,y,w,h,id=box_id
        cv2.putText(roi,str(id),(x,y-15),cv2.FONT_HERSHEY_PLAIN,2,(255,0,0),2)
        cv2.rectangle(roi,(x,y),(x+w,y+h),(0,255,0),3)
    
    cv2.imshow("ROI",roi)
    cv2.imshow("Frame",frame)
    cv2.imshow("Mask",mask)

    

    key=cv2.waitKey(10)

    if key==ord('q') or key == 27:
        break


cap.release()
cv2.destroyAllWindows()

✅ Video opened successfully
{0: (458, 288)}
{0: (458, 300)}
{0: (457, 310)}
{0: (456, 322)}
{0: (456, 334)}
{0: (456, 346)}
{0: (455, 359)}
{0: (454, 374)}
{0: (453, 388)}
{0: (452, 404)}
{0: (451, 420)}
{0: (450, 432)}
{0: (448, 441)}
{0: (445, 450)}
{1: (513, 206)}
{1: (514, 211)}
{1: (513, 216)}
{1: (514, 222)}
{1: (514, 227)}
{1: (514, 232)}
{1: (514, 238)}
{1: (515, 245)}
{1: (515, 252)}
{1: (516, 255)}
{1: (517, 261)}
{1: (517, 269)}
{1: (520, 276)}
{1: (517, 282)}
{1: (518, 288)}
{1: (517, 296)}
{1: (518, 303)}
{1: (517, 311)}
{1: (520, 318), 2: (445, 241)}
{1: (520, 318), 2: (445, 247)}
{1: (520, 327), 2: (445, 247)}
{1: (520, 327), 2: (444, 255)}
{1: (521, 335), 2: (444, 255)}
{1: (521, 335), 2: (444, 264)}
{1: (521, 343), 2: (444, 264)}
{1: (521, 343), 2: (442, 274)}
{1: (522, 352), 2: (442, 274)}
{1: (522, 352), 2: (441, 282)}
{1: (522, 361), 2: (441, 282)}
{1: (522, 361), 2: (446, 293)}
{1: (523, 370), 2: (446, 293)}
{1: (523, 370), 2: (439, 303)}
{1: (524, 380), 2: (439, 3

In [12]:
import cv2
import numpy as np
from tracker import Euclideandisttraker

# -----------------------------
# Initialize Tracker
# -----------------------------
tracker = Euclideandisttraker()

cap = cv2.VideoCapture(
    r"C:\Users\beher\OneDrive\Desktop\DeepLearning\Computer vision\tracker img\highway.mp4"
)

if not cap.isOpened():
    print("❌ Video could not be opened!")
    exit()

# -----------------------------
# Background Subtractor
# -----------------------------
object_detector = cv2.createBackgroundSubtractorMOG2(
    history=100,
    varThreshold=40,
    detectShadows=False
)

# -----------------------------
# Counting Variables
# -----------------------------
counter = []
line_y = 230
offset = 8

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # -----------------------------
    # ROI
    # -----------------------------
    roi = frame[250:720, 200:1100]

    # Counting Line
    cv2.line(
        roi,
        (0, line_y),
        (roi.shape[1], line_y),
        (0, 0, 255),
        2
    )

    # -----------------------------
    # Foreground Mask
    # -----------------------------
    mask = object_detector.apply(roi)

    _, mask = cv2.threshold(mask, 250, 255, cv2.THRESH_BINARY)

    kernel = np.ones((5, 5), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    # -----------------------------
    # Find Contours
    # -----------------------------
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_TREE,
        cv2.CHAIN_APPROX_SIMPLE
    )

    detections = []

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area > 500:

            x, y, w, h = cv2.boundingRect(cnt)

            detections.append([x, y, w, h])

    # -----------------------------
    # Tracking
    # -----------------------------
    boxes_ids = tracker.update(detections)

    for box_id in boxes_ids:

        x, y, w, h, id = box_id

        cx = x + w // 2
        cy = y + h // 2

        cv2.rectangle(
            roi,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        cv2.putText(
            roi,
            str(id),
            (x, y - 10),
            cv2.FONT_HERSHEY_PLAIN,
            2,
            (255, 0, 0),
            2
        )

        cv2.circle(
            roi,
            (cx, cy),
            4,
            (0, 0, 255),
            -1
        )

        # -----------------------------
        # Counting Logic
        # -----------------------------
        if line_y - offset < cy < line_y + offset:

            if id not in counter:
                counter.append(id)

                # Green line when counted
                cv2.line(
                    roi,
                    (0, line_y),
                    (roi.shape[1], line_y),
                    (0, 255, 0),
                    3
                )

    # -----------------------------
    # Display Count
    # -----------------------------
    cv2.putText(
        roi,
        f"Count : {len(counter)}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        2
    )

    cv2.imshow("ROI", roi)
    cv2.imshow("Mask", mask)
    cv2.imshow("Frame", frame)

    key = cv2.waitKey(20)

    if key == ord("q") or key == 27:
        break

cap.release()
cv2.destroyAllWindows()

{1: (466, 117), 2: (37, 10)}
{2: (34, 11)}
{2: (30, 12)}
{2: (26, 13), 3: (465, 138)}
{3: (464, 144), 2: (26, 13)}
{3: (464, 144), 2: (23, 23)}
{3: (464, 144), 2: (22, 16)}
{2: (20, 17), 4: (464, 154)}
{4: (466, 159), 2: (20, 17)}
{4: (466, 159), 2: (18, 17)}
{4: (465, 166), 2: (18, 17)}
{4: (465, 166), 2: (19, 18)}
{4: (464, 173), 2: (19, 18)}
{4: (464, 173), 2: (17, 19)}
{4: (464, 181), 2: (17, 19)}
{4: (464, 181), 2: (15, 21)}
{4: (464, 186), 2: (15, 21)}
{4: (464, 186), 2: (11, 19)}
{4: (463, 195), 2: (11, 19)}
{5: (481, 231)}
{5: (478, 235), 6: (18, 314)}
{6: (18, 314), 5: (479, 251)}
{5: (477, 258)}
{5: (477, 267)}
{5: (476, 278)}
{5: (478, 289)}
{5: (476, 296), 7: (31, 298)}
{7: (20, 310), 5: (476, 296)}
{7: (20, 310), 5: (475, 308)}
{7: (20, 310), 5: (476, 320)}
{5: (470, 327)}
{5: (470, 340), 8: (17, 315)}
{8: (17, 315), 5: (472, 355)}
{5: (473, 367)}
{5: (474, 382), 9: (23, 309)}
{5: (474, 382), 9: (23, 309)}
{5: (472, 397), 9: (23, 309)}
{5: (472, 397), 9: (22, 310)}
{5: (46